# Brain Code Notebook Six: Comprehensive Decision Lab

This notebook starts the deeper decision-practice work. It is not Notebook Two.

Notebook Two was an introduction. Notebook Six asks for new tasks and observations: many runs, fair comparisons, spread, averages, and final evidence.

Student rhythm:

1. Predict
2. Run
3. Look
4. Explain

Sentence frame:

> When I changed ___, the graph changed by ___. I think this means ___.

## What Is New Here?

Notebook Six is comprehensive because you will:

- rerun the same settings
- compare many paths
- change the decision bound
- change noise
- match plots to questions
- choose final evidence

## Setup

Run this cell once. You do not need to edit it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['font.size'] = 12

def run_decision(evidence_push=0.05, noise=0.18, bound=1.0, max_steps=80, seed=None):
    rng = np.random.default_rng(seed)
    evidence = 0.0
    path = [evidence]
    choice = 'no choice yet'
    for step in range(1, max_steps + 1):
        evidence += evidence_push + rng.normal(0, noise)
        path.append(evidence)
        if evidence >= bound:
            choice = 'Choose A'
            break
        if evidence <= -bound:
            choice = 'Choose B'
            break
    return np.arange(len(path)), np.array(path), choice, len(path) - 1

def plot_decision_path(evidence_push=0.05, noise=0.18, bound=1.0, seed=1, title='One decision path'):
    t, path, choice, rt = run_decision(evidence_push=evidence_push, noise=noise, bound=bound, seed=seed)
    plt.figure()
    plt.plot(t, path, linewidth=2)
    plt.axhline(bound, linestyle='--', color='green', label='Choose A bound')
    plt.axhline(-bound, linestyle='--', color='crimson', label='Choose B bound')
    plt.axhline(0, color='gray', linewidth=1)
    plt.xlabel('step')
    plt.ylabel('evidence')
    plt.title(title)
    plt.legend(loc='best')
    plt.show()
    print('Choice:', choice)
    print('Decision time:', rt, 'steps')
    return t, path, choice, rt

def run_many(n=40, evidence_push=0.05, noise=0.18, bound=1.0, seed=10):
    choices = []
    times = []
    paths = []
    for i in range(n):
        t, path, choice, rt = run_decision(evidence_push=evidence_push, noise=noise, bound=bound, seed=seed + i)
        choices.append(choice)
        times.append(rt)
        paths.append((t, path))
    return choices, np.array(times), paths

def summarize_many(choices, times):
    choose_a = choices.count('Choose A')
    choose_b = choices.count('Choose B')
    total = len(choices)
    print('Choose A:', choose_a, 'of', total, f'({100 * choose_a / total:.0f}%)')
    print('Choose B:', choose_b, 'of', total, f'({100 * choose_b / total:.0f}%)')
    print('Average decision time:', round(float(np.mean(times)), 1), 'steps')
    print('Fastest / slowest:', int(np.min(times)), '/', int(np.max(times)), 'steps')

def plot_many_paths(paths, bound=1.0, title='Many decision paths'):
    plt.figure()
    for t, path in paths[:20]:
        plt.plot(t, path, alpha=0.35)
    plt.axhline(bound, linestyle='--', color='green')
    plt.axhline(-bound, linestyle='--', color='crimson')
    plt.axhline(0, color='gray', linewidth=1)
    plt.xlabel('step')
    plt.ylabel('evidence')
    plt.title(title)
    plt.show()

def compare_setting(label, values, setting_name):
    rows = []
    for value in values:
        kwargs = dict(evidence_push=0.05, noise=0.18, bound=1.0)
        kwargs[setting_name] = value
        choices, times, paths = run_many(n=40, seed=20, **kwargs)
        choose_a = choices.count('Choose A')
        rows.append((value, choose_a, round(float(np.mean(times)), 1), int(np.min(times)), int(np.max(times))))
    print(label)
    print('value | Choose A count | avg time | fastest | slowest')
    print('-' * 58)
    for value, choose_a, avg_time, fastest, slowest in rows:
        print(f'{value:>5} | {choose_a:>14} | {avg_time:>8} | {fastest:>7} | {slowest:>7}')
    return rows

## Baseline: Many Runs

Predict: Will every run look exactly the same?

Run: Use the same settings many times.

Look: Compare choices and decision times.

Explain: Same settings can still make a spread of paths.

In [ ]:
baseline_choices, baseline_times, baseline_paths = run_many(n=40, evidence_push=0.05, noise=0.18, bound=1.0, seed=10)
plot_many_paths(baseline_paths, bound=1.0, title='Baseline: many decision paths')
summarize_many(baseline_choices, baseline_times)

## Activity 13: Rerun Same Settings

Predict: If settings stay the same, will the summary stay close?

Run: Rerun with a new seed.

Look: Compare the two summaries.

Explain: When I changed ___, the graph changed by ___. I think this means ___.

In [ ]:
rerun_choices, rerun_times, rerun_paths = run_many(n=40, evidence_push=0.05, noise=0.18, bound=1.0, seed=100)
plot_many_paths(rerun_paths, bound=1.0, title='Rerun: same settings, new samples')
print('First summary')
summarize_many(baseline_choices, baseline_times)
print()
print('Second summary')
summarize_many(rerun_choices, rerun_times)

### Activity 13 Observation

Write one observation:

- What stayed similar?
- What changed a little?
- Why is one run not enough?

## Activity 14: Change The Bound

Predict: What happens when the model waits for more evidence?

Run: Compare low, normal, and high bound.

Look: Use average time and choice count.

Explain: When I changed ___, the graph changed by ___. I think this means ___.

In [ ]:
bound_rows = compare_setting('Bound comparison', [0.7, 1.0, 1.3], 'bound')

### Activity 14 Observation

Use the table.

Sentence starter:

> A higher bound changed ___ because ___.

## Activity 15: Change Noise

Predict: What happens when wobble gets stronger?

Run: Compare low, normal, and high noise.

Look: Compare path spread and fastest / slowest time.

Explain: When I changed ___, the graph changed by ___. I think this means ___.

In [ ]:
noise_rows = compare_setting('Noise comparison', [0.08, 0.18, 0.30], 'noise')
low_choices, low_times, low_paths = run_many(n=40, evidence_push=0.05, noise=0.08, bound=1.0, seed=30)
high_choices, high_times, high_paths = run_many(n=40, evidence_push=0.05, noise=0.30, bound=1.0, seed=30)
plot_many_paths(low_paths, bound=1.0, title='Low noise paths')
plot_many_paths(high_paths, bound=1.0, title='High noise paths')

### Activity 15 Observation

Use the graph and table.

- Which setting looked more variable?
- Which setting had a wider fastest / slowest range?

## Activity 16: Which Plot Answers Which Question?

Predict: Which graph would answer your question best?

Run: Look at summaries and paths.

Look: Match question to graph.

Explain: When I changed ___, the graph changed by ___. I think this means ___.

In [ ]:
questions = {
    'Which choice happened more often?': 'choice percent / count',
    'How fast was the decision?': 'average decision time',
    'How variable were the paths?': 'many-path plot or fastest / slowest range',
}
for question, graph in questions.items():
    print(question, '->', graph)

## Activity 17: Choose Final Decision Evidence

Choose one graph or table from Notebook Six.

Use one evidence type:

- choice count
- average time
- path spread
- fastest / slowest range

In [ ]:
my_evidence_type = '___'
what_i_changed = '___'
what_graph_showed = '___'
what_i_think = '___'
one_limitation = '___'

print('Evidence type:', my_evidence_type)
print('I changed:', what_i_changed)
print('The graph showed:', what_graph_showed)
print('I think this means:', what_i_think)
print('One limitation:', one_limitation)

## Mini-Share Sentence

Use the final frame:

> I changed ___. The graph showed ___. I think this means ___. One limitation is ___.

## What Makes Notebook Six More Complete?

Notebook Six goes beyond the first look:

- many runs, not one run
- same settings rerun
- bound comparison
- noise comparison
- graph-question matching
- final evidence choice